# 04 · Human-in-the-loop — the review siding

Why documents land there, what waits on disk, and how a decision re-enters the graph.

## Setup — the lab bench

In [1]:
import json
import sys
from pathlib import Path

# Work from the repo root no matter where the kernel was started.
ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
assert (ROOT / "notebooks" / "pipeline_lab.py").exists(), (
    f"llm-mailroom repo root not found above {Path.cwd()}"
)
sys.path.insert(0, str(ROOT / "notebooks"))

import pipeline_lab as lab


## What you'll see

- a run parked at `human_review` (low-confidence classification)
- the pending review record on disk
- resuming the SAME thread with an approval -> archived
- rejecting -> the FAILED bin (rejection is terminal, not rework)
- the boss escalation route

**Honesty label:** real graph + real checkpointer thread; mock LLM.

## Park two documents

In [2]:
env4 = lab.open_sandbox()
r_ok = lab.run_to_review(env4, lab.DOC_CONTRACT, filename="approve_me.txt")
r_rj = lab.run_to_review(env4, lab.DOC_CONTRACT, filename="reject_me.txt")
DOC_ID = r_ok["final"]["doc_id"]
RJ_ID = r_rj["final"]["doc_id"]
for r in (r_ok, r_rj):
    f = r["final"]
    print(f"{f['original_filename']:15s} stage={f['stage']:<7} "
          f"reason={f.get('escalation_reason')}")


approve_me.txt  stage=review  reason=Unsure
reject_me.txt   stage=review  reason=Unsure


## What waits on disk

In [3]:
recs = lab.artifacts(Path(env4["base_dir"]))
print("review bin:", recs["review_bin"])


review bin: ['reject_me.txt', 'approve_me.txt']


## Approve — the thread continues

In [4]:
r2 = lab.resume_review(env4, DOC_ID, "approve_me.txt")
f2 = r2["final"]
print("decision -> stage:", f2.get("review_decision"), "->", f2["stage"])
print("path tail:", lab.path_of(r2["steps"]))


decision -> stage: approved -> archived
path tail: ['extract-fields', 'compile-report', 'write-catalog', 'archive-document']


## Reject — terminal, honestly

In [5]:
r3 = lab.reject_review(env4, RJ_ID)
for k in ("stage", "review_decision"):
    print(f"{k}:", r3[k])


stage: PipelineStage.FAILED
review_decision: rejected


Rejection does NOT loop the document back for another attempt: the
router (`after_human_review`) treats any non-approval as terminal and the
run ends in the FAILED bin with the rejection recorded. Re-work means a
human fixes the source and lets the watcher pick it up as a fresh run.

## Bench teardown

In [6]:
lab.close_sandbox(env4)
print("sandbox closed")


sandbox closed


In [7]:
with lab.lab_sandbox() as env:
    lab.script_client(env["client"], boss=lab.BOSS_REVIEW)
    print("boss canned decision:", lab.BOSS_REVIEW["decision"])
    print("-> after_boss routes any non-'approved' ruling to human_review")


boss canned decision: review
-> after_boss routes any non-'approved' ruling to human_review


## Where to go next

- **06 · outputs_and_audit** — the paper trail these runs left
- **05 · failure_recovery** — the other way runs end